## Dependencies

In [ ]:
## libraries
import sys
import logging
import importlib
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.estimators.factories import load_estimators
from src.data.builders import (
    load_processed_data,
    load_perturbed_data
)
from src.evaluators.perturbing import (
    eval_perturbed,
    stat_perturbed_test,
    stat_perturbed_summary
)
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z,
    TARGET
)

## Initalization

In [2]:
## load data and models
_disable = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    data = load_processed_data()
    data_pert = load_perturbed_data()
    models = load_estimators()
finally:
    logging.disable(_disable)

## view data shape
n_obs, n_feat = data.shape
print("Original Data")
print(f"    Features: {n_feat}")
print(f"    Observations: {n_obs}")

print("\nPerturbed Data")
for json_key, methods in data_pert.items():
    n_methods = len(methods)
    n_settings = sum(len(intensities) for intensities in methods.values())
    print(f"  {json_key}:")
    print(f"    Methods: {n_methods}")
    print(f"    Settings: {n_settings}")

Original Data
    Features: 32
    Observations: 25

Perturbed Data
  invariants_perturbed:
    Methods: 3
    Settings: 21
  network_perturbed:
    Methods: 3
    Settings: 21
  process_perturbed:
    Methods: 3
    Settings: 21
  signatures_perturbed:
    Methods: 3
    Settings: 21
  temporal_perturbed:
    Methods: 3
    Settings: 21


## Perturbation Evaluation
This section evaluates whether frontier metrics degrade under perturbation. Under the frozen protocol, models trained on original data are evaluated on perturbed inputs. Under the retrain protocol, models are retrained on perturbed data and compared to the original-data baseline.

In [ ]:
## perturbation evaluation (retrain + frozen logo-cv)
if "results_perturbed" not in globals():
    results_perturbed, perturbed_all, recovery_df = eval_perturbed(
        data = data,
        models = models,
        data_pert = data_pert,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
    )

## Perturbation Family Tests
This section tests whether each perturbation family collapses the frontier by lowering EI. Each table runs a paired one-sided Wilcoxon signed-rank test comparing baseline EI against perturbed EI within a single perturbation family. Under this framing, significant results indicate evidence of frontier collapse for that family, while non-significant results indicate no detected collapse under that class of perturbation.

## Network Perturbation Test
This section tests whether network perturbations collapse the frontier by lowering EI.

In [4]:
## paired wilcoxon test for network perturbation EI
results_table_network = stat_perturbed_test(
    results = results_perturbed,
    feat_value = ["ei"],
    feat_group = ["track", "method"],
    pert_type = "network"
)

display(results_table_network)

=== Perturbation: Original vs Perturbed Median EI (n = 63) ===
H₁: Perturbation lowers median EI
*** p < 0.001, ** p < 0.01, * p < 0.05


Median EI (Original) Median EI (Perturbed)  \
Track   Method                                                           
frozen  network:node_sample               0.7512                0.7416   
        network:rewire                    0.7512                0.7291   
        network:sparsify                  0.7512                0.7541   
retrain network:node_sample               0.7512                0.7144   
        network:rewire                    0.7512                0.7308   
        network:sparsify                  0.7512                0.7185   

                            Median Δ  EI Positive Δ  Wilcoxon W+  \
Track   Method                                                     
frozen  network:node_sample       0.0096      0.7302        1198   
        network:rewire            0.0221      0.7778        1360   
        network:sparsify         -0.0029      0.7302        1243   
retrain network:node_sample       0.0368      0.5556        1038   
        network:rewire            0.0204      0.5556        1053   
        network:sparsify          0.0327      0.5556         954   

                            Rank-biserial r One-sided p Holm-adjusted p Sig  
Track   Method                                                               
frozen  network:node_sample          0.1885      0.0965          0.3862      
        network:rewire               0.3492      0.0079          0.0477   *  
        network:sparsify             0.2331      0.0537          0.2687      
retrain network:node_sample          0.0298      0.4186             1.0      
        network:rewire               0.0446       0.379             1.0      
        network:sparsify            -0.0536      0.6443             1.0

## Invariant Perturbation Test
This section tests whether invariant perturbations collapse the frontier by lowering EI.

In [5]:
## paired wilcoxon test for invariant perturbation EI
results_table_invariants = stat_perturbed_test(
    results = results_perturbed,
    feat_value = ["ei"],
    feat_group = ["track", "method"],
    pert_type = "invariants"
)

display(results_table_invariants)

=== Perturbation: Original vs Perturbed Median EI (n = 63) ===
H₁: Perturbation lowers median EI
*** p < 0.001, ** p < 0.01, * p < 0.05


Median EI (Original) Median EI (Perturbed)  \
Track   Method                                                         
frozen  invariants:jitter               0.7512                0.7363   
        invariants:noise                0.7512                 0.742   
        invariants:subset               0.7512                0.7465   
retrain invariants:jitter               0.7512                0.7189   
        invariants:noise                0.7512                0.7052   
        invariants:subset               0.7512                0.7319   

                          Median Δ  EI Positive Δ  Wilcoxon W+  \
Track   Method                                                   
frozen  invariants:jitter       0.0148      0.6825        1071   
        invariants:noise        0.0092       0.619         941   
        invariants:subset       0.0047      0.6825        1038   
retrain invariants:jitter       0.0323      0.5397         974   
        invariants:noise         0.046      0.5397         980   
        invariants:subset       0.0193      0.6032        1028   

                          Rank-biserial r One-sided p Holm-adjusted p Sig  
Track   Method                                                             
frozen  invariants:jitter          0.0625       0.333             1.0      
        invariants:noise          -0.0665      0.6769             1.0      
        invariants:subset          0.0298      0.4186             1.0      
retrain invariants:jitter         -0.0337      0.5921             1.0      
        invariants:noise          -0.0278       0.576             1.0      
        invariants:subset          0.0198      0.4455             1.0

## Process Perturbation Test
This section tests whether process perturbations collapse the frontier by lowering EI.

In [6]:
## paired wilcoxon test for process perturbation EI
results_table_process = stat_perturbed_test(
    results = results_perturbed,
    feat_value = ["ei"],
    feat_group = ["track", "method"],
    pert_type = "process"
)

display(results_table_process)

=== Perturbation: Original vs Perturbed Median EI (n = 63) ===
H₁: Perturbation lowers median EI
*** p < 0.001, ** p < 0.01, * p < 0.05


Median EI (Original) Median EI (Perturbed)  \
Track   Method                                                             
frozen  process:bootstrapping               0.7512                0.7292   
        process:scaling                     0.7512                0.7144   
        process:smoothing                   0.7512                0.7566   
retrain process:bootstrapping               0.7512                 0.741   
        process:scaling                     0.7512                 0.721   
        process:smoothing                   0.7512                0.7208   

                              Median Δ  EI Positive Δ  Wilcoxon W+  \
Track   Method                                                       
frozen  process:bootstrapping        0.022      0.4286         921   
        process:scaling             0.0368      0.5556        1158   
        process:smoothing          -0.0054      0.2698         434   
retrain process:bootstrapping       0.0101      0.4127         679   
        process:scaling             0.0302      0.4762        1030   
        process:smoothing           0.0304      0.5238         910   

                              Rank-biserial r One-sided p Holm-adjusted p Sig  
Track   Method                                                                 
frozen  process:bootstrapping         -0.0863      0.7243             1.0      
        process:scaling                0.1488      0.1522          0.9134      
        process:smoothing             -0.5694         1.0             1.0      
retrain process:bootstrapping         -0.3264      0.9879             1.0      
        process:scaling                0.0218      0.4401             1.0      
        process:smoothing             -0.0972      0.7489             1.0

## Signature Perturbation Test
This section tests whether signature perturbations collapse the frontier by lowering EI.

In [7]:
## paired wilcoxon test for signature perturbation EI
results_table_signatures = stat_perturbed_test(
    results = results_perturbed,
    feat_value = ["ei"],
    feat_group = ["track", "method"],
    pert_type = "signatures"
)

display(results_table_signatures)

=== Perturbation: Original vs Perturbed Median EI (n = 63) ===
H₁: Perturbation lowers median EI
*** p < 0.001, ** p < 0.01, * p < 0.05


Median EI (Original) Median EI (Perturbed)  \
Track   Method                                                         
frozen  signatures:jitter               0.7512                0.7417   
        signatures:noise                0.7512                0.7428   
        signatures:subset               0.7512                0.7283   
retrain signatures:jitter               0.7512                 0.728   
        signatures:noise                0.7512                0.7289   
        signatures:subset               0.7512                0.7104   

                          Median Δ  EI Positive Δ  Wilcoxon W+  \
Track   Method                                                   
frozen  signatures:jitter       0.0095      0.5238         837   
        signatures:noise        0.0083      0.5397         859   
        signatures:subset       0.0229      0.6984        1046   
retrain signatures:jitter       0.0232      0.5079        1017   
        signatures:noise        0.0223      0.4127         924   
        signatures:subset       0.0408      0.4603         889   

                          Rank-biserial r One-sided p Holm-adjusted p Sig  
Track   Method                                                             
frozen  signatures:jitter         -0.1696      0.8791             1.0      
        signatures:noise          -0.1478      0.8462             1.0      
        signatures:subset          0.0377      0.3973             1.0      
retrain signatures:jitter          0.0089      0.4754             1.0      
        signatures:noise          -0.0833      0.7174             1.0      
        signatures:subset         -0.1181      0.7926             1.0

## Temporal Perturbation Test
This section tests whether temporal perturbations collapse the frontier by lowering EI.

In [8]:
## paired wilcoxon test for temporal perturbation EI
results_table_temporal = stat_perturbed_test(
    results = results_perturbed,
    feat_value = ["ei"],
    feat_group = ["track", "method"],
    pert_type = "temporal"
)

display(results_table_temporal)

=== Perturbation: Original vs Perturbed Median EI (n = 63) ===
H₁: Perturbation lowers median EI
*** p < 0.001, ** p < 0.01, * p < 0.05


Median EI (Original) Median EI (Perturbed)  \
Track   Method                                                            
frozen  temporal:aggregation               0.7512                 0.683   
        temporal:dropout                   0.7512                0.7547   
        temporal:jitter                    0.7512                0.6719   
retrain temporal:aggregation               0.7512                0.7492   
        temporal:dropout                   0.7512                0.7411   
        temporal:jitter                    0.7512                0.7755   

                             Median Δ  EI Positive Δ  Wilcoxon W+  \
Track   Method                                                      
frozen  temporal:aggregation       0.0682      0.7778        1721   
        temporal:dropout          -0.0035      0.4762         782   
        temporal:jitter            0.0793      0.8571        1886   
retrain temporal:aggregation       0.0019      0.2857         563   
        temporal:dropout           0.0101      0.5397         933   
        temporal:jitter           -0.0243      0.2222         212   

                             Rank-biserial r One-sided p Holm-adjusted p  Sig  
Track   Method                                                                 
frozen  temporal:aggregation          0.7073         0.0             0.0  ***  
        temporal:dropout             -0.2242      0.9391             1.0       
        temporal:jitter                0.871         0.0             0.0  ***  
retrain temporal:aggregation         -0.4415      0.9988             1.0       
        temporal:dropout             -0.0744      0.6962             1.0       
        temporal:jitter              -0.7897         1.0             1.0

In [9]:
## median perturbation deltas by family
delta_summary = stat_perturbed_summary(
    results = perturbed_all,
    feat_group = ["track", "perturbation"]
)

display(delta_summary)

Δ VR    Δ MV    Δ MS    Δ EA    Δ EI
track   perturbation                                      
frozen  invariants    0.00  0.0000  0.0710  0.0303  0.0020
        network       0.00  0.0000  0.0710  0.0331  0.0027
        process       0.00 -0.1013  0.1267  0.0487 -0.0110
        signatures    0.00  0.0000  0.0986  0.0354  0.0020
        temporal      0.04  0.1909  0.0546 -0.0195  0.0146
retrain invariants    0.00 -0.1038  0.1725  0.0457  0.0104
        network       0.00  0.0191  0.1165  0.0457  0.0104
        process       0.00 -0.0175  0.4127  0.0855 -0.0031
        signatures    0.00 -0.0232  0.1404  0.0457 -0.0043
        temporal      0.00 -0.1038  0.9408  0.0244 -0.0191

## Recovery Ratios
This section compares the degree to which retraining recovers performance lost under perturbation. Recovery ratio is defined as the fraction of frozen-track degradation that retraining eliminates, computed only where frozen degradation is meaningfully positive.

In [10]:
## median recovery ratios
recovery_summary = stat_perturbed_summary(
    results = recovery_df,
    feat_group = ["perturbation"]
)

display(recovery_summary)

,ρ VR,ρ MV,ρ MS,ρ EA,ρ EI
perturbation,,,,,
invariants,0.8706,0.0062,-0.1340,-0.3809,0.7233
network,1.5385,1.3246,0.3128,-0.1147,1.6132
process,1.0000,0.7091,0.4100,0.2879,0.7372
signatures,0.2500,0.1169,0.1807,-0.1826,0.1312
temporal,1.0000,1.0962,0.9500,0.9085,1.1812
